# Real-Time Crisis Detection System
## Notebook 02 — Text Preprocessing and NER
**Goal:** Clean all raw text, normalize Indonesian slang, detect language, run NER to extract locations/times/orgs, match crisis keywords, and save `posts_cleaned.csv`.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/10Academy/crisis-detection-system'
elif IN_KAGGLE:
    PROJECT_DIR = '/kaggle/working/crisis-detection-system'
else:
    PROJECT_DIR = os.path.abspath('..')

DATA_PROCESSED = f'{PROJECT_DIR}/data/processed'
DATA_EXTERNAL  = f'{PROJECT_DIR}/data/external'
UTILS_DIR      = f'{PROJECT_DIR}/utils'
if UTILS_DIR not in sys.path: sys.path.insert(0, UTILS_DIR)
print(f' Ready. Project: {PROJECT_DIR}')

Mounted at /content/drive
✅ Ready. Project: /content/drive/MyDrive/10Academy/crisis-detection-system


In [ ]:
!pip install -q spacy langdetect
!python -m spacy download en_core_web_sm -q
!python -m spacy download xx_ent_wiki_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 75.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Load posts_unified.csv
import pandas as pd
import numpy as np
import json, re, ast
from tqdm import tqdm
tqdm.pandas()

from nlp_utils import (
    clean_text, normalize_indonesian_slang, detect_language,
    run_ner, match_crisis_keywords, compute_metadata_features,
    compute_linguistic_uncertainty, compute_relevance_flag,
    INDONESIAN_SLANG
)

unified_path = f'{DATA_PROCESSED}/posts_unified.csv'
df = pd.read_csv(unified_path)
df = df.set_index('post_id')

print(f' Loaded posts_unified.csv: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Null counts:\n{df.isnull().sum()}')

✅ Loaded posts_unified.csv: (75, 10)
Columns: ['source_platform', 'raw_text', 'timestamp_raw', 'user_id', 'location_raw', 'geo_lat', 'geo_lon', 'media_present', 'engagement_score', 'label']
Null counts:
source_platform      0
raw_text             0
timestamp_raw        0
user_id             55
location_raw        43
geo_lat             60
geo_lon             60
media_present        0
engagement_score     0
label               41
dtype: int64


In [ ]:
# Text encoding normalization
import html, unicodedata

def normalize_encoding(text):
    if not isinstance(text, str): return ''
    text = html.unescape(text)
    text = unicodedata.normalize('NFC', text)
# Remove zero-width spaces
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
# Normalize curly quotes and em-dashes to ASCII
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    return text.strip()

df['raw_text'] = df['raw_text'].apply(normalize_encoding)
print(f' Encoding normalization done. Sample:')
print(df['raw_text'].iloc[0][:120])

✅ Encoding normalization done. Sample:
HOI IV | Open Beta Weekend (1.19.2)                  Generals!   We decided it would be best to run 1.19.2 as an Open Be


In [ ]:
# Platform-specific noise removal
def apply_platform_cleaning(row):
    platform = str(row.get('source_platform', 'twitter'))
    return clean_text(str(row['raw_text']), platform=platform)

df['text_cleaned'] = df.progress_apply(apply_platform_cleaning, axis=1)

# Drop rows where cleaning removed everything
before = len(df)
df = df[df['text_cleaned'].str.len() >= 10]
print(f' Platform cleaning done. Dropped {before - len(df)} empty rows.')
print(f'Sample cleaned text:')
print(df['text_cleaned'].iloc[0][:150])

100%|██████████| 75/75 [00:00<00:00, 2099.62it/s]

✅ Platform cleaning done. Dropped 0 empty rows.
Sample cleaned text:
HOI IV | Open Beta Weekend (1.19.2) Generals! We decided it would be best to run 1.19.2 as an Open Beta over the weekend to get some potential player 


In [ ]:
# Indonesian slang normalization
# Try loading the full slang dict from external file
slang_path = f'{DATA_EXTERNAL}/indonesian_slang_dict.json'
try:
    with open(slang_path) as f:
        slang_dict = json.load(f)
    slang_dict.update(INDONESIAN_SLANG)  # merge with built-in
    print(f' Loaded slang dictionary: {len(slang_dict)} entries')
except FileNotFoundError:
    slang_dict = INDONESIAN_SLANG
    print(f'Using built-in slang dict: {len(slang_dict)} entries')

df['text_cleaned'] = df['text_cleaned'].apply(
    lambda t: normalize_indonesian_slang(t, slang_dict)
)
print(' Indonesian slang normalization done.')
# Show before/after example for a post with slang
slang_posts = df[df['source_platform'].isin(['twitter','telegram'])]
if not slang_posts.empty:
    example = slang_posts['text_cleaned'].iloc[0]
    print(f'Example: {example[:120]}')

Using built-in slang dict: 31 entries
✅ Indonesian slang normalization done.
Example: [BNPB] Gempa M5.6 guncang Cianjur Jawa Barat pada 21 Nov 2022 pukul 13:21 WIB. Dilaporkan ada korban dan kerusakan bangu


In [ ]:
# Language detection
print('Detecting languages... (this may take a moment)')
df['lang_detected'] = df['text_cleaned'].progress_apply(detect_language)

print('\nLanguage distribution:')
print(df['lang_detected'].value_counts().to_string())

# Flag low-quality 'other' language posts (no crisis keywords)
other_posts = df[df['lang_detected'] == 'other']
print(f'\nPosts tagged "other" language: {len(other_posts)}')
print('These are retained but low priority.')

Detecting languages... (this may take a moment)


100%|██████████| 75/75 [00:01<00:00, 38.60it/s]


Language distribution:
lang_detected
id    46
en    29

Posts tagged "other" language: 0
These are retained but low priority.


In [ ]:
# & 7: Tokenization + NER
# Load a simple Indonesian place name list for gazetteer-backed NER
INDONESIA_PLACES = [
    'Jakarta', 'Surabaya', 'Bandung', 'Medan', 'Semarang', 'Makassar',
    'Palembang', 'Tangerang', 'Depok', 'Bekasi', 'Bogor', 'Yogyakarta',
    'Solo', 'Malang', 'Pekanbaru', 'Banjarmasin', 'Pontianak', 'Bali',
    'Denpasar', 'Aceh', 'Cianjur', 'Ciawi', 'Kemang', 'Lombok', 'Palu',
    'Donggala', 'Luwu', 'Wamena', 'Papua', 'Kupang', 'Manado', 'Ambon',
    'Jayapura', 'Samarinda', 'Balikpapan', 'Mataram', 'Tasikmalaya',
    'Subang', 'Ciamis', 'Garut', 'Sukabumi', 'Banten', 'Serang',
    'Lampung', 'Padang', 'Jambi', 'Bengkulu', 'Palu', 'Kendari',
    'Sulawesi', 'Kalimantan', 'Sumatera', 'Jawa', 'Flores', 'Timor',
    'Gunung Merapi', 'Gunung Semeru', 'Gunung Agung', 'Anak Krakatau',
    'Sungai Ciliwung', 'Tol Cipularang', 'Selat Sunda',
]

def extract_ner_for_row(row):
    text = str(row['text_cleaned'])
    lang = str(row['lang_detected'])
    try:
        entities = run_ner(text, lang=lang, gazetteer=INDONESIA_PLACES)
    except Exception:
        entities = {'LOC': [], 'DATE': [], 'TIME': [], 'ORG': []}
    return (
        json.dumps(entities.get('LOC', []), ensure_ascii=False),
        json.dumps(entities.get('DATE', []) + entities.get('TIME', []), ensure_ascii=False),
        json.dumps(entities.get('ORG', []), ensure_ascii=False),
        len(entities.get('LOC', [])) + len(entities.get('DATE', [])) + len(entities.get('ORG', []))
    )

print('Running NER... (may take 1-2 minutes for large datasets)')
ner_results = df.progress_apply(extract_ner_for_row, axis=1)
df['entities_location'] = [r[0] for r in ner_results]
df['entities_time']     = [r[1] for r in ner_results]
df['entities_org']      = [r[2] for r in ner_results]
df['entity_count']      = [r[3] for r in ner_results]

print('\n NER complete.')
print(f'Posts with ≥1 location entity: {(df["entities_location"] != "[]").sum()}')
print(f'Posts with ≥1 time entity: {(df["entities_time"] != "[]").sum()}')

Running NER... (may take 1-2 minutes for large datasets)


100%|██████████| 75/75 [00:21<00:00,  3.46it/s]


✅ NER complete.
Posts with ≥1 location entity: 58
Posts with ≥1 time entity: 18


In [ ]:
# Crisis keyword matching
from nlp_utils import CRISIS_KEYWORDS_ID, CRISIS_KEYWORDS_EN, URGENCY_SIGNALS

# Try to load external lexicons
try:
    with open(f'{DATA_EXTERNAL}/crisis_lexicon_id.txt') as f:
        lex_id = [l.strip() for l in f if l.strip()]
    with open(f'{DATA_EXTERNAL}/crisis_lexicon_en.txt') as f:
        lex_en = [l.strip() for l in f if l.strip()]
    print(f' External lexicons: {len(lex_id)} ID + {len(lex_en)} EN terms')
except FileNotFoundError:
    lex_id, lex_en = CRISIS_KEYWORDS_ID, CRISIS_KEYWORDS_EN
    print(f'Using built-in lexicons: {len(lex_id)+len(lex_en)} terms')

def apply_keyword_match(text):
    matched, count, urgency = match_crisis_keywords(str(text), lex_en, lex_id)
    return json.dumps(matched, ensure_ascii=False), count, urgency

kw_results = df['text_cleaned'].progress_apply(apply_keyword_match)
df['crisis_keywords_found'] = [r[0] for r in kw_results]
df['keyword_hit_count']     = [r[1] for r in kw_results]
df['urgency_signal']        = [r[2] for r in kw_results]

print(f'\n Keyword matching done.')
print(f'Posts with ≥1 keyword hit: {(df["keyword_hit_count"] >= 1).sum()}')
print(f'Posts with urgency signals: {df["urgency_signal"].sum()}')

Using built-in lexicons: 38 terms


100%|██████████| 75/75 [00:00<00:00, 3239.08it/s]


✅ Keyword matching done.
Posts with ≥1 keyword hit: 70
Posts with urgency signals: 15


In [ ]:
# Metadata feature computation
def compute_meta(row):
    text = str(row['text_cleaned'])
    m = compute_metadata_features(
        text,
        has_media=bool(row.get('media_present', False)),
        engagement_score=float(row.get('engagement_score', 0) or 0)
    )
    return pd.Series(m)

meta_df = df.progress_apply(compute_meta, axis=1)
df = pd.concat([df, meta_df], axis=1)

print(' Metadata features computed:')
print(df[['text_length','token_count','has_media',
          'contains_number','contains_url']].describe())

100%|██████████| 75/75 [00:00<00:00, 1285.27it/s]

✅ Metadata features computed:
        text_length  token_count
count     75.000000    75.000000
mean    3429.720000   553.613333
std     7802.444252  1283.681308
min       57.000000     9.000000
25%       76.000000    11.000000
50%      184.000000    24.000000
75%      585.000000    92.000000
max    30112.000000  5252.000000


In [ ]:
# Linguistic uncertainty + Relevance flag
df['linguistic_uncertainty'] = df['text_cleaned'].apply(
    lambda t: compute_linguistic_uncertainty(str(t))
)

df['relevance_flag'] = df.apply(
    lambda r: compute_relevance_flag(
        r['keyword_hit_count'], r['entity_count'], r['urgency_signal']
    ), axis=1
)

print(f' Relevance flagging done.')
print(f'Relevant posts (flag=1): {df["relevance_flag"].sum()} / {len(df)}')
print(f'Avg linguistic uncertainty: {df["linguistic_uncertainty"].mean():.3f}')

✅ Relevance flagging done.
Relevant posts (flag=1): 58 / 75
Avg linguistic uncertainty: 0.010


In [ ]:
# Inspect and validate NER outputs
import random
print('=== NER SAMPLE VALIDATION ===')
sample = df[df['entities_location'] != '[]'].sample(
    min(10, len(df[df['entities_location'] != '[]'])), random_state=42
)
for idx, row in sample.iterrows():
    locs = json.loads(row['entities_location'])
    print(f'\n[{idx}] [{row["source_platform"]}]')
    print(f'  Text: {str(row["text_cleaned"])[:100]}')
    print(f'  Locations: {locs}')
    print(f'  Keywords: {json.loads(row["crisis_keywords_found"])}')
    print(f'  Lang: {row["lang_detected"]} | Relevance: {row["relevance_flag"]}')

=== NER SAMPLE VALIDATION ===

[RD_6110537505215153603] [reddit]
  Text: HOI IV | Open Beta Weekend (1.19.2) Generals! We decided it would be best to run 1.19.2 as an Open B
  Locations: ['+15%', 'Indonesia', 'East Timor', 'Netherlands', 'Amsterdam', 'Jayapura', 'Australasia', 'Australia', 'Singapore', 'UK', 'Japan', 'AI', 'Iceland', 'Greenland', 'Romania', "Puppet Romania'", 'Bulgaria', 'the Republic of Indonesia', 'Bandung', 'Nauru', 'Philippines', 'Papua', 'Germany', 'Iraq', 'Hungary', 'Britain', 'the United Kingdom', 'Iran', 'Extrême', 'Netherlands-Indonesia Union', 'France', 'Milford', 'Ireland', 'Africa', 'Bougainville', 'pacific islands', 'Spain', 'Fixes', 'Ethiopia', 'the Horn of Africa', 'Aceh', 'Timor']
  Keywords: ['fire', 'flood']
  Lang: en | Relevance: 1

[RD_1437634991234081175] [reddit]
  Text: "Godzilla" El Niño May Have Begun, Here's What This Means for Earth This video, presented by Anton P
  Locations: ['Earth', 'Asia', 'Australia', 'South America', 'Indonesia', 'No

In [ ]:
# Save posts_cleaned.csv
output_cols = [
    'source_platform','raw_text','text_cleaned','timestamp_raw',
    'user_id','location_raw','geo_lat','geo_lon',
    'media_present','engagement_score','label',
    'lang_detected','entities_location','entities_time','entities_org',
    'entity_count','keyword_hit_count','crisis_keywords_found',
    'urgency_signal','relevance_flag','linguistic_uncertainty',
    'text_length','token_count','has_media','contains_number',
    'contains_url','url_domain'
]
# Only keep columns that exist
output_cols = [c for c in output_cols if c in df.columns]

save_path = f'{DATA_PROCESSED}/posts_cleaned.csv'
df[output_cols].to_csv(save_path)

print(f' Saved posts_cleaned.csv: {len(df)} rows → {save_path}')
print(f'\nColumn list saved: {output_cols}')
print(f'\n PREPROCESSING COMPLETE ')
print(f'Total rows: {len(df)}')
print(f'Relevant rows (flag=1): {df["relevance_flag"].sum()}')
print(f'Language distribution: {df["lang_detected"].value_counts().to_dict()}')
print(f'\n Next: Run 03_geo_temporal_inference.ipynb')

✅ Saved posts_cleaned.csv: 75 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/processed/posts_cleaned.csv

Column list saved: ['source_platform', 'raw_text', 'text_cleaned', 'timestamp_raw', 'user_id', 'location_raw', 'geo_lat', 'geo_lon', 'media_present', 'engagement_score', 'label', 'lang_detected', 'entities_location', 'entities_time', 'entities_org', 'entity_count', 'keyword_hit_count', 'crisis_keywords_found', 'urgency_signal', 'relevance_flag', 'linguistic_uncertainty', 'text_length', 'token_count', 'has_media', 'contains_number', 'contains_url', 'url_domain']

═══ PREPROCESSING COMPLETE ═══
Total rows: 75
Relevant rows (flag=1): 58
Language distribution: {'id': 46, 'en': 29}

👉 Next: Run 03_geo_temporal_inference.ipynb
